# 15 - Analysis Workflow (Capstone)


A realistic data-analysis loop in pure NumPy:

1. **Generate / load** data.
2. **Inspect** shape, dtypes, basic stats.
3. **Clean** — drop invalid values, fill gaps (`np.nanmean`, masks).
4. **Transform** — add computed columns (vectorised).
5. **Filter & group** — boolean masks and `np.unique` for categories.
6. **Aggregate & report** — reductions, percentiles, correlations.
7. **Persist** results with `savetxt` / `savez`.

This notebook runs the whole pipeline on a synthetic **sales dataset**.

> Everything below is plain NumPy — no pandas — so you see the machinery behind the curtain.

In [ ]:
import numpy as np
import os

rng = np.random.default_rng(7)

## 1. Generate a synthetic dataset

10 000 sales rows: `region`, `units`, `price`, and some missing `units` values.

In [ ]:
n = 10_000
regions = np.array(["north", "south", "east", "west"])
region = rng.choice(regions, n)
units = rng.integers(1, 20, n).astype("float64")
units[rng.random(n) < 0.05] = np.nan          # 5% missing
price = np.round(rng.uniform(5, 50, n), 2)

sales = np.column_stack([region, units, price])
print("shape:", sales.shape)

## 2. Inspect

First rows, dtypes, and quick stats per column.

In [ ]:
sales[:5]

In [ ]:
print("regions:", np.unique(region, return_counts=True))
print("units non-missing:", np.count_nonzero(~np.isnan(units)), "missing:", np.count_nonzero(np.isnan(units)))
print("price range:", price.min(), "-", price.max())

## 3. Clean

Decide a strategy: **fill missing units** with the median (median is robust to outliers).

In [ ]:
median_units = np.nanmedian(units)
units = np.where(np.isnan(units), median_units, units)
print("missing now:", np.count_nonzero(np.isnan(units)))

## 4. Transform

Add a computed column: `revenue = units * price`.

In [ ]:
revenue = units * price
print("total revenue :", revenue.sum().round(2))
print("mean per order:", revenue.mean().round(2))

## 5. Filter & group

Boolean masks extract subsets; `np.unique` groups by region.

In [ ]:
big_orders = revenue[revenue > 500]
print(f"orders > $500: {len(big_orders)}  avg value: {big_orders.mean().round(2)}")

In [ ]:
# Revenue by region, the vectorised way
regions_u = np.unique(region)
for r in regions_u:
    mask = region == r
    print(f"{r:>6}  orders={mask.sum():5d}  revenue={revenue[mask].sum():10,.0f}")

## 6. Aggregate & report

Percentiles, averages, and correlation tell the story of the data.

In [ ]:
print("revenue quartiles:", np.percentile(revenue, [0, 25, 50, 75, 100]).round(2))
print("revenue mean    :", revenue.mean().round(2))
print("revenue std     :", revenue.std().round(2))

In [ ]:
corr = np.corrcoef(units, price)[0, 1]
print(f"corr(units, price) = {corr:.3f}  (near 0 => independent)")

## 7. Persist results

Save the cleaned data and the computed revenue column for later use.

In [ ]:
clean = np.column_stack([units, price, revenue])
np.savetxt("sales_clean.csv", clean, delimiter=",", header="units,price,revenue", comments="", fmt="%.2f")
np.savez("sales_summary.npz", regions=regions_u, revenue=revenue)
print(open("sales_clean.csv").read()[:80], "...")
print("saved sales_clean.csv and sales_summary.npz")

In [ ]:
# Clean up the generated files
for f in ["sales_clean.csv", "sales_summary.npz"]:
    if os.path.exists(f):
        os.remove(f)
print("generated files removed")